# Практическая работа 2. Часть 1

В ноутбуке исследуются основные способы локального запуска языковых моделей и влияние квантизации на их производительность.

В первом блоке сравниваются **Transformers, llama.cpp, Ollama и LM Studio** по удобству, скорости, гибкости и сложности использования. Во втором блоке исследуется модель `Qwen2.5-1.5B-Instruct` в квантизациях **Q4, Q5 и Q8**: сравниваются время загрузки, потребление памяти, скорость генерации и качество ответов.


### Задание 1. Сравнение способов локального запуска модели

В этом эксперименте сравниваются четыре способа локального запуска LLM:

* Hugging Face Transformers;
* llama.cpp;
* Ollama;
* LM Studio.

Во всех случаях используется одна базовая модель — `Qwen2.5-1.5B-Instruct`. Для llama.cpp, Ollama и LM Studio используется квантизованная версия Q8, а Transformers работает с исходным checkpoint Hugging Face.

Для каждого инструмента фиксируются время установки и настройки, размер модели, наличие OpenAI-совместимого API и производительность на одинаковом prompt. Генерация выполняется три раза, после чего рассчитываются средние значения TTFT, общего времени и tokens/sec.

Следует учитывать, что LM Studio запускается локально на Windows-компьютере, а остальные инструменты — в Google Colab, поэтому абсолютная скорость между этими средами напрямую не сопоставима.


In [7]:
# Общий промпт для всех четырёх инструментов

PROMPT = """
Представь, что ты объясняешь начинающему ML-инженеру,
как выбрать между локальным запуском языковой модели
и использованием облачного API.

Сравни эти подходы с точки зрения:
1. конфиденциальности данных;
2. стоимости;
3. производительности;
4. сложности инфраструктуры;
5. масштабирования.

В конце приведи один практический пример, когда локальная модель
будет предпочтительнее облачной, и один пример обратной ситуации.

Ответ дай связным текстом на русском языке.
"""

#### Этап 1. Transformers

In [1]:
# Замерим время установки

import time

start = time.perf_counter()

!pip install -q transformers accelerate

transformers_install_time = time.perf_counter() - start

print(f"Время установки Transformers: {transformers_install_time:.2f} сек.")

Время установки Transformers: 10.24 сек.


In [2]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:
# Теперь отдельно замерим загрузку модели

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

model_load_time = time.perf_counter() - start

print(f"Время загрузки модели: {model_load_time:.2f} сек.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Время загрузки модели: 60.29 сек.


In [4]:
# Размер модели

from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()

for repo in cache_info.repos:
    if repo.repo_id == MODEL_ID:
        print(
            f"Размер модели в HF cache: "
            f"{repo.size_on_disk / 1024**3:.2f} GB"
        )

Размер модели в HF cache: 2.89 GB


Измеряем TTFT, total time и tokens/sec.

Для TTFT нужна потоковая генерация. У Transformers для этого можно использовать TextIteratorStreamer.

In [6]:
from transformers import TextIteratorStreamer
from threading import Thread

In [7]:
def benchmark_transformers(prompt, max_new_tokens=200):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt")

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
    }

    start = time.perf_counter()

    thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs,
    )

    thread.start()

    first_token_time = None
    generated_text = ""

    for chunk in streamer:
        if first_token_time is None:
            first_token_time = time.perf_counter()

        generated_text += chunk

    thread.join()

    end = time.perf_counter()

    ttft = first_token_time - start
    total_time = end - start

    generated_tokens = len(
        tokenizer.encode(
            generated_text,
            add_special_tokens=False,
        )
    )

    tokens_per_second = generated_tokens / total_time

    return {
        "ttft": ttft,
        "total_time": total_time,
        "generated_tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
        "answer": generated_text,
    }

Выполняем три запуска, как этого требует условие задания, и считаем средние.

In [8]:
transformers_results = []

for i in range(3):

    result = benchmark_transformers(PROMPT)
    transformers_results.append(result)

    print(f"\nЗапуск {i + 1}")
    print(f"TTFT: {result['ttft']:.2f} сек.")
    print(f"Total time: {result['total_time']:.2f} сек.")
    print(
        f"Speed: "
        f"{result['tokens_per_second']:.2f} tokens/sec"
    )


Запуск 1
TTFT: 12.61 сек.
Total time: 161.65 сек.
Speed: 1.24 tokens/sec

Запуск 2
TTFT: 9.70 сек.
Total time: 153.35 сек.
Speed: 1.30 tokens/sec

Запуск 3
TTFT: 10.56 сек.
Total time: 153.18 сек.
Speed: 1.31 tokens/sec


In [9]:
import statistics


transformers_summary = {
    "tool": "Transformers",
    "avg_ttft": statistics.mean(
        r["ttft"]
        for r in transformers_results
    ),
    "avg_total_time": statistics.mean(
        r["total_time"]
        for r in transformers_results
    ),
    "avg_tokens_per_second": statistics.mean(
        r["tokens_per_second"]
        for r in transformers_results
    ),
}

In [10]:
transformers_summary

{'tool': 'Transformers',
 'avg_ttft': 10.955214071,
 'avg_total_time': 156.05788245500017,
 'avg_tokens_per_second': 1.2823848119444647}

Итого по transformers:

- установка: 10.24 с
- загрузка модели: 60.29 с
- размер: 2.89 GB
- средний TTFT: 10.96 с
- среднее полное время: 156.06 с
- средняя скорость: 1.28 tok/s
- OpenAI-compatible API: нет

#### Этап 2. llama.cpp

Примечание: сборка llama.cpp из исходников оказалась слишком долгой, поэтому я использовал простой вариант с готовым CPU wheel.

In [1]:
# Замерим время установки

import time

start = time.perf_counter()

!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

llama_install_time = time.perf_counter() - start

print(f"Время установки llama-cpp-python: {llama_install_time:.2f} сек.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
Время установки llama-cpp-python: 19.22 сек.


Оставляем ту же модель:

In [2]:
!pip install -q huggingface_hub

In [3]:
from huggingface_hub import hf_hub_download
import os
import time

GGUF_REPO = "bartowski/Qwen2.5-1.5B-Instruct-GGUF"
GGUF_FILE = "Qwen2.5-1.5B-Instruct-Q8_0.gguf"

start = time.perf_counter()

GGUF_PATH = hf_hub_download(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
)

download_time = time.perf_counter() - start

model_size_gb = os.path.getsize(GGUF_PATH) / 1024**3

print(f"Время скачивания модели: {download_time:.2f} сек.")
print(f"Размер GGUF: {model_size_gb:.2f} GB")

Qwen2.5-1.5B-Instruct-Q8_0.gguf: reconstructing file:   0%|          |  0.00B / 1.65GB            

Qwen2.5-1.5B-Instruct-Q8_0.gguf: downloading bytes:           |  0.00B            

Время скачивания модели: 18.74 сек.
Размер GGUF: 1.53 GB


Загружаем модель через llama.cpp:

In [4]:
from llama_cpp import Llama

start = time.perf_counter()

llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=2048,
    n_threads=2,
    verbose=False,
)

llama_load_time = time.perf_counter() - start

print(f"Время загрузки модели: {llama_load_time:.2f} сек.")

Время загрузки модели: 5.54 сек.


Напишем функцию для бенчмарка llama.cpp, выполним три запуска и посчитаем статистику.

In [5]:
def benchmark_llama_cpp(prompt, max_tokens=200):
    start = time.perf_counter()

    first_token_time = None
    generated_text = ""

    stream = llm.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        max_tokens=max_tokens,
        temperature=0,
        stream=True,
    )

    generated_tokens = 0

    for chunk in stream:
        delta = chunk["choices"][0]["delta"]
        content = delta.get("content")

        if content:
            if first_token_time is None:
                first_token_time = time.perf_counter()

            generated_text += content

    end = time.perf_counter()

    ttft = first_token_time - start
    total_time = end - start

    # Считаем токены средствами самой llama.cpp-модели
    generated_tokens = len(
        llm.tokenize(
            generated_text.encode("utf-8"),
            add_bos=False,
        )
    )

    tokens_per_second = generated_tokens / total_time

    return {
        "ttft": ttft,
        "total_time": total_time,
        "generated_tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
        "answer": generated_text,
    }

In [8]:
import statistics

llama_results = []

for i in range(3):
    result = benchmark_llama_cpp(PROMPT)

    llama_results.append(result)

    print(f"\nЗапуск {i + 1}")
    print(f"TTFT: {result['ttft']:.2f} сек.")
    print(f"Total time: {result['total_time']:.2f} сек.")
    print(
        f"Speed: "
        f"{result['tokens_per_second']:.2f} tokens/sec"
    )


Запуск 1
TTFT: 14.25 сек.
Total time: 106.25 сек.
Speed: 1.88 tokens/sec

Запуск 2
TTFT: 0.97 сек.
Total time: 71.07 сек.
Speed: 2.81 tokens/sec

Запуск 3
TTFT: 0.27 сек.
Total time: 72.95 сек.
Speed: 2.74 tokens/sec


In [9]:
llama_summary = {
    "tool": "llama.cpp",
    "avg_ttft": statistics.mean(
        r["ttft"] for r in llama_results
    ),
    "avg_total_time": statistics.mean(
        r["total_time"] for r in llama_results
    ),
    "avg_tokens_per_second": statistics.mean(
        r["tokens_per_second"]
        for r in llama_results
    ),
}

llama_summary

{'tool': 'llama.cpp',
 'avg_ttft': 5.164996755666607,
 'avg_total_time': 83.41985629166668,
 'avg_tokens_per_second': 2.4794609006803796}

Итого по llama.cpp:

- установка: 19.22 с
- скачивание модели: 18.74 с
- размер модели: 1.53 GB
- загрузка модели: 5.54 с
- средний TTFT: 5.16 с
- среднее полное время: 83.42 с
- средняя скорость: 2.48 tok/s
- OpenAI-compatible API: да, через llama-server
- в Colab использовал Python binding llama-cpp-python

Примечание: есть заметный эффект прогрева: первый запуск был существенно медленнее (TTFT 14.25 с), а второй и третий уже около 1 секунды и ниже.

#### Этап 3. Ollama

In [12]:
!sudo apt install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (331 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122809 files and director

In [13]:
import time

start = time.perf_counter()

!curl -fsSL https://ollama.com/install.sh | sh

ollama_install_time = time.perf_counter() - start

print(f"Время установки Ollama: {ollama_install_time:.2f} сек.")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Время установки Ollama: 39.53 сек.


Запуск сервера:

In [15]:
import subprocess
import time

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(3)

print("Ollama server запущен")

Ollama server запущен


Скачиваем модель и подгружаем её:

In [17]:
start = time.perf_counter()

!ollama pull qwen2.5:1.5b-instruct-q8_0

ollama_download_time = time.perf_counter() - start

print(f"Время скачивания модели: {ollama_download_time:.2f} сек.")


Время скачивания модели: 37.59 сек.


In [18]:
!ollama list

NAME                          ID              SIZE      MODIFIED               
qwen2.5:1.5b-instruct-q8_0    b19683a34698    1.6 GB    Less than a second ago    


Ollama поддерживает OpenAI-style endpoint /v1/chat/completions:

In [19]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
)

Напишем функцию для оценки модели и проведём три запуска, посчитав статистики:

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [20]:
def benchmark_ollama(prompt, max_tokens=200):
    start = time.perf_counter()
    first_token_time = None

    generated_text = ""

    stream = ollama_client.chat.completions.create(
        model="qwen2.5:1.5b-instruct-q8_0",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        max_tokens=max_tokens,
        temperature=0,
        stream=True,
    )

    for chunk in stream:
        content = chunk.choices[0].delta.content

        if content:
            if first_token_time is None:
                first_token_time = time.perf_counter()

            generated_text += content

    end = time.perf_counter()

    ttft = first_token_time - start
    total_time = end - start

    generated_tokens = len(
        tokenizer.encode(
            generated_text,
            add_special_tokens=False,
        )
    )

    tokens_per_second = generated_tokens / total_time

    return {
        "ttft": ttft,
        "total_time": total_time,
        "generated_tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
        "answer": generated_text,
    }

In [22]:
import statistics

ollama_results = []

for i in range(3):
    result = benchmark_ollama(PROMPT)

    ollama_results.append(result)

    print(f"\nЗапуск {i + 1}")
    print(f"TTFT: {result['ttft']:.2f} сек.")
    print(f"Total time: {result['total_time']:.2f} сек.")
    print(
        f"Speed: "
        f"{result['tokens_per_second']:.2f} tokens/sec"
    )


Запуск 1
TTFT: 42.98 сек.
Total time: 111.40 сек.
Speed: 1.80 tokens/sec

Запуск 2
TTFT: 0.29 сек.
Total time: 71.00 сек.
Speed: 2.82 tokens/sec

Запуск 3
TTFT: 0.35 сек.
Total time: 64.42 сек.
Speed: 3.10 tokens/sec


In [23]:
ollama_summary = {
    "tool": "Ollama",
    "avg_ttft": statistics.mean(
        r["ttft"] for r in ollama_results
    ),
    "avg_total_time": statistics.mean(
        r["total_time"] for r in ollama_results
    ),
    "avg_tokens_per_second": statistics.mean(
        r["tokens_per_second"]
        for r in ollama_results
    ),
}

ollama_summary

{'tool': 'Ollama',
 'avg_ttft': 14.539063288999992,
 'avg_total_time': 82.27315451433317,
 'avg_tokens_per_second': 2.572268492085034}

#### Этап 4. LM Studio

**Примечание.** Этап с LM Studio выполнялся отдельно на локальном Windows-компьютере, поскольку LM Studio является desktop-приложением. Модель `Qwen2.5-1.5B-Instruct-Q8_0` была загружена в LM Studio и запущена через встроенный локальный OpenAI-compatible API. Замеры TTFT, общего времени генерации и tokens/sec выполнялись отдельным Python-скриптом, который я привожу ниже.

In [ ]:
import time
import statistics

from openai import OpenAI


BASE_URL = "http://127.0.0.1:1234/v1"

# Идентификатор модели из LM Studio
MODEL_ID = "qwen2.5-1.5b-instruct"

MAX_TOKENS = 200
N_RUNS = 3

# Тот же prompt, который использовался в Colab

PROMPT = """
Представь, что ты объясняешь начинающему ML-инженеру,
как выбрать между локальным запуском языковой модели
и использованием облачного API.

Сравни эти подходы с точки зрения:
1. конфиденциальности данных;
2. стоимости;
3. производительности;
4. сложности инфраструктуры;
5. масштабирования.

В конце приведи один практический пример, когда локальная модель
будет предпочтительнее облачной, и один пример обратной ситуации.

Ответ дай связным текстом на русском языке.
"""

# Используем клиент OpenAI

client = OpenAI(base_url=BASE_URL, api_key="lm-studio")

print("Проверка OpenAI-compatible API...")

models = client.models.list()

print("API работает.")
print("Доступные модели:")

for model in models.data:
    print("-", model.id)

# Функция для запуска бенчмарка

def benchmark_lmstudio(prompt, max_tokens=200):
    start = time.perf_counter()
    first_token_time = None

    generated_text = ""
    generated_tokens = None

    stream = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        max_tokens=max_tokens,
        temperature=0,
        stream=True,

        # LM Studio вернёт usage в последнем streaming chunk
        stream_options={
            "include_usage": True
        },
    )

    for chunk in stream:

        # В последнем chunk LM Studio может вернуть usage
        if chunk.usage is not None:
            generated_tokens = chunk.usage.completion_tokens

        if not chunk.choices:
            continue

        content = chunk.choices[0].delta.content

        if content:
            if first_token_time is None:
                first_token_time = time.perf_counter()

            generated_text += content

    end = time.perf_counter()

    ttft = first_token_time - start
    total_time = end - start

    if generated_tokens is None:
        raise RuntimeError(
            "LM Studio не вернул completion_tokens в streaming usage."
        )

    tokens_per_second = generated_tokens / total_time

    return {
        "ttft": ttft,
        "total_time": total_time,
        "generated_tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
        "answer": generated_text,
    }

# Запускаем три эксперимента и считаем статистику

results = []

for i in range(N_RUNS):
    result = benchmark_lmstudio(
        prompt=PROMPT,
        max_tokens=MAX_TOKENS,
    )

    results.append(result)

    print(f"\nЗапуск {i + 1}")
    print(f"TTFT: {result['ttft']:.2f} сек.")
    print(f"Total time: {result['total_time']:.2f} сек.")
    print(f"Generated tokens: {result['generated_tokens']}")
    print(
        f"Speed: {result['tokens_per_second']:.2f} tokens/sec"
    )


summary = {
    "tool": "LM Studio",
    "avg_ttft": statistics.mean(
        r["ttft"] for r in results
    ),
    "avg_total_time": statistics.mean(
        r["total_time"] for r in results
    ),
    "avg_tokens_per_second": statistics.mean(
        r["tokens_per_second"] for r in results
    ),
}

print("\n" + "=" * 60)
print("СРЕДНИЕ РЕЗУЛЬТАТЫ")
print("=" * 60)

print(summary)

print("\nПример ответа модели:\n")
print(results[0]["answer"])

Итого по LM Studio:

- время установки программы: ≈ 5 мин
- время загрузки модели: ≈ 2 мин
- размер модели по LM Studio: 1.89 GB
- средний TTFT: 0.79 с
- среднее полное время: 26.56 с
- средняя скорость: 7.53 tok/s
- OpenAI-compatible API: да

#### Сравнительная таблица:

| Инструмент   | Размер модели | Средний TTFT | Среднее время |   Скорость | OpenAI API | Удобство      | Гибкость | Сложность    |
| ------------ | ------------: | -----------: | ------------: | ---------: | ---------- | ------------- | -------- | ------------ |
| Transformers |       2.89 GB |      10.96 с |      156.06 с | 1.28 tok/s | Нет        | Среднее       | Высокая  | Средняя      |
| llama.cpp    |       1.53 GB |       5.16 с |       83.42 с | 2.48 tok/s | Да         | Среднее       | Высокая  | Выше средней |
| Ollama       |        1.6 GB |      14.54 с |       82.27 с | 2.57 tok/s | Да         | Высокое       | Средняя  | Низкая       |
| LM Studio    |       1.89 GB |       0.79 с |       26.56 с | 7.53 tok/s | Да         | Очень высокое | Средняя  | Низкая       |


Примечание: LM Studio тестировался на отдельном Windows-компьютере, а остальные инструменты — в CPU-среде Google Colab, поэтому абсолютные показатели производительности LM Studio не сравниваются напрямую с остальными результатами.

### Анализ и вывод

На CPU в Colab `llama.cpp` и Ollama показали близкую скорость — около 2.5 токена/с — и заметно опередили Transformers (1.28 токена/с). У Ollama и llama.cpp первый запрос выполнялся медленнее последующих, что связано с первоначальной загрузкой и прогревом модели.

LM Studio показал наиболее высокую скорость и минимальный TTFT, однако он запускался на отдельном локальном Windows-компьютере, поэтому напрямую сравнивать его производительность с Colab некорректно.

По удобству разработки наиболее простыми оказались **LM Studio и Ollama**: они быстро позволяют загрузить модель, запустить локальный сервер и использовать OpenAI-совместимый API. **Transformers** удобен для экспериментов и глубокого контроля над моделью, но сам по себе не предоставляет серверный API. **llama.cpp** требует немного больше настройки, зато предоставляет высокий уровень контроля, эффективную работу с GGUF и подходит для развёртывания локальных моделей с минимальными зависимостями.

Таким образом, для прототипирования и разработки наиболее удобны LM Studio или Ollama, для исследовательской работы — Transformers, а для более контролируемого и ресурсно-эффективного production-развёртывания — llama.cpp.

### Задание 2. Влияние квантизации в llama.cpp

В эксперименте сравниваются три версии одной модели Qwen2.5-1.5B-Instruct с разной степенью квантизации: Q4, Q5 и Q8.

Для каждой версии измеряются время загрузки, потребление RAM, скорость генерации и время ответа на длинный запрос. Затем модели выполняют одинаковые задания на логику, генерацию кода и пересказ текста, после чего сравниваются качество ответов и производительность.

In [ ]:
# !pip install -q llama-cpp-python \
#     --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

# !pip install -q huggingface_hub psutil

In [25]:
import gc
import os
import time
import psutil

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

Скачиваем все три модели:

In [26]:
REPO_ID = "bartowski/Qwen2.5-1.5B-Instruct-GGUF"

MODEL_FILES = {
    "Q4": "Qwen2.5-1.5B-Instruct-Q4_K_M.gguf",
    "Q5": "Qwen2.5-1.5B-Instruct-Q5_K_M.gguf",
    "Q8": "Qwen2.5-1.5B-Instruct-Q8_0.gguf",
}

model_paths = {}

for quant, filename in MODEL_FILES.items():

    path = hf_hub_download(repo_id=REPO_ID, filename=filename)
    model_paths[quant] = path
    size_gb = os.path.getsize(path) / 1024**3

    print(f"{quant}: {size_gb:.2f} GB")

Qwen2.5-1.5B-Instruct-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B /  986MB            

Qwen2.5-1.5B-Instruct-Q4_K_M.gguf: downloading bytes:           |  0.00B            

Q4: 0.92 GB


Qwen2.5-1.5B-Instruct-Q5_K_M.gguf: reconstructing file:   0%|          |  0.00B / 1.13GB            

Qwen2.5-1.5B-Instruct-Q5_K_M.gguf: downloading bytes:           |  0.00B            

Q5: 1.05 GB
Q8: 1.53 GB


Будем измерять увеличение RAM процесса после загрузки модели:

In [27]:
def get_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2


def load_model(model_path):
    gc.collect()

    ram_before = get_ram_mb()

    start = time.perf_counter()

    llm = Llama(model_path=model_path, n_ctx=4096, n_threads=2, verbose=False)

    load_time = time.perf_counter() - start

    ram_after = get_ram_mb()

    return llm, {
        "load_time": load_time,
        "ram_mb": ram_after - ram_before,
    }

In [28]:
def generate_and_measure(llm, prompt, max_tokens=200):
    start = time.perf_counter()

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        max_tokens=max_tokens,
        temperature=0,
    )

    total_time = time.perf_counter() - start

    text = response["choices"][0]["message"]["content"]

    generated_tokens = len(
        llm.tokenize(
            text.encode("utf-8"),
            add_bos=False,
        )
    )

    tokens_per_second = generated_tokens / total_time

    return {
        "time": total_time,
        "tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
        "answer": text,
    }

Короткий prompt для скорости:

In [29]:
SPEED_PROMPT = """
Кратко объясни начинающему ML-инженеру,
что такое квантизация языковой модели,
зачем она используется и какой компромисс
возникает между размером модели и качеством ответа.
"""

Длинный запрос 500+ токенов:

In [30]:
context_block = """
Локальные языковые модели позволяют выполнять inference без передачи данных
во внешний облачный сервис. При выборе способа запуска необходимо учитывать
размер модели, доступную оперативную память, скорость процессора, наличие GPU,
требования к конфиденциальности, стоимость инфраструктуры и предполагаемую
нагрузку. Квантизация позволяет уменьшить размер весов модели и потребление
памяти за счёт хранения параметров с меньшей точностью. Более сильная
квантизация обычно повышает эффективность использования ресурсов, но может
в некоторой степени снижать качество генерации. При практическом выборе важно
оценивать не только скорость, но и качество ответов на типичных задачах.
"""

LONG_PROMPT = (
    context_block * 8
    + """
На основе приведённого контекста объясни основные преимущества и недостатки
квантизации локальных LLM и сформулируй практическую рекомендацию по выбору
между Q4, Q5 и Q8.
"""
)

In [31]:
# проверка, что промпт удовлетворяет условию задачи

len(
    llm.tokenize(
        LONG_PROMPT.encode("utf-8"),
        add_bos=False,
    )
)

1622

Три задания на качество:

In [32]:
LOGIC_PROMPT = """
У Маши, Олега и Иры разные профессии:
врач, программист и учитель.

Известно:
- Маша не врач.
- Олег не программист.
- Ира не учитель.
- Олег не врач.

Определи профессию каждого и кратко объясни рассуждение.
"""

CODE_PROMPT = """
Напиши функцию на Python:

def find_duplicates(values):

Она должна принимать список целых чисел и возвращать
отсортированный список значений, которые встречаются
более одного раза.

Пример:
find_duplicates([3, 1, 2, 3, 2, 5]) -> [2, 3]

Добавь краткое объяснение решения.
"""

SUMMARY_TEXT = """
Компания разрабатывала систему обработки документов вручную.
Каждый поступающий документ сотрудник открывал, определял его тип,
извлекал необходимые поля и переносил информацию в базу данных.
С ростом числа документов ручная обработка стала занимать слишком
много времени и приводить к ошибкам. Команда решила внедрить систему
автоматической классификации и извлечения данных. После запуска
пилотной версии время обработки сократилось, но оказалось, что для
редких типов документов качество распознавания было ниже. Поэтому
команда добавила ручную проверку для случаев с низкой уверенностью
модели и начала регулярно дообучать систему на новых примерах.
"""

SUMMARY_PROMPT = f"""
Перескажи следующий текст в 3–4 предложениях,
сохранив основные факты и причинно-следственные связи.

Текст:
{SUMMARY_TEXT}
"""

Один цикл для всех Q4/Q5/Q8:

In [33]:
results = {}

for quant, path in model_paths.items():

    print(f"\n{'=' * 60}")
    print(quant)
    print("=" * 60)

    llm, load_stats = load_model(path)

    print(
        f"Load time: {load_stats['load_time']:.2f} сек."
    )
    print(
        f"RAM increase: {load_stats['ram_mb']:.0f} MB"
    )

    speed_result = generate_and_measure(
        llm,
        SPEED_PROMPT,
        max_tokens=200,
    )

    long_result = generate_and_measure(
        llm,
        LONG_PROMPT,
        max_tokens=200,
    )

    logic_result = generate_and_measure(
        llm,
        LOGIC_PROMPT,
        max_tokens=200,
    )

    code_result = generate_and_measure(
        llm,
        CODE_PROMPT,
        max_tokens=250,
    )

    summary_result = generate_and_measure(
        llm,
        SUMMARY_PROMPT,
        max_tokens=200,
    )

    results[quant] = {
        "load": load_stats,
        "speed": speed_result,
        "long": long_result,
        "logic": logic_result,
        "code": code_result,
        "summary": summary_result,
    }

    print(
        f"Speed: "
        f"{speed_result['tokens_per_second']:.2f} tok/s"
    )

    print(
        f"Long prompt time: "
        f"{long_result['time']:.2f} сек."
    )

    # Освобождаем модель перед загрузкой следующей
    del llm
    gc.collect()


Q4
Load time: 15.86 сек.
RAM increase: 1684 MB
Speed: 1.55 tok/s
Long prompt time: 242.62 сек.

Q5
Load time: 8.35 сек.
RAM increase: 1179 MB
Speed: 3.17 tok/s
Long prompt time: 297.36 сек.

Q8
Load time: 12.87 сек.
RAM increase: 1677 MB
Speed: 2.96 tok/s
Long prompt time: 220.65 сек.


In [34]:
import pandas as pd

rows = []

for quant, r in results.items():
    rows.append({
        "quantization": quant,
        "load_time_sec": r["load"]["load_time"],
        "ram_mb": r["load"]["ram_mb"],
        "tokens_per_sec": r["speed"]["tokens_per_second"],
        "long_prompt_time_sec": r["long"]["time"],
    })

metrics_df = pd.DataFrame(rows)

metrics_df

,quantization,load_time_sec,ram_mb,tokens_per_sec,long_prompt_time_sec
0,Q4,15.859599,1684.355469,1.546883,242.623249
1,Q5,8.353870,1179.300781,3.166564,297.359248
2,Q8,12.873004,1676.730469,2.964451,220.645661


Выводим ответы для оценки качества:

In [35]:
for quant, r in results.items():

    print(f"\n{'=' * 80}")
    print(quant)
    print("=" * 80)

    print("\nLOGIC:")
    print(r["logic"]["answer"])

    print("\nCODE:")
    print(r["code"]["answer"])

    print("\nSUMMARY:")
    print(r["summary"]["answer"])


Q4

LOGIC:
Давайте разберем эту задачу шаг за шагом:

1) Мы знаем, что:
   - Маша не врач
   - Олег не программист
   - Ира не учитель
   - Олег не врач

2) Мы также знаем, что:
   - Маша и Ира не программисты
   - Олег и Ира не учители

3) Пусть Маша - это врач. Тогда:
   - Маша не врач (противоречие)
   - Олег не программист (противоречие)
   - Ира не учитель (противоречие)

4) Теперь давайте посмотрим, что означает "Олег не врач":
   - Если Маша - врач, то Олег не может быть врачом.

5) Пусть Маша - врач, тогда:
   - Олег

CODE:
Функция `find_duplicates` принимает список целых чисел и возвращает отсортированный список значений, которые встречаются более одного раза. Вот краткий объяснение решения:

1. **Сортировка списка**: Мы сначала сортируем список, чтобы упростить поиск повторяющихся элементов.

2. **Итерация по списку**: Мы идем по каждому элементу списка и проверяем, есть ли он в списке уже просмотренных элементов.

3. **Замена элемента**: Если элемент встречается более одног

Оценка проводилась вручную по трём критериям:

- логика — правильное решение и связность рассуждения;
- код — корректность алгоритма и соответствие условию;
- пересказ — сохранение основных фактов без искажений.

По каждой задаче ставится балл:

- 0 — неправильный ответ
- 1 — частично правильный
- 2 — полностью правильный

| Квантизация | Логика                                                        | Код                                                                                         | Пересказ   | Итог |
| ----------- | ------------------------------------------------------------- | ------------------------------------------------------------------------------------------- | ---------- | ---: |
| Q4          | Ошибочное рассуждение, ответ не завершён                      | Решение не реализовано, ответ обрывается                                                    | Корректный |  2/6 |
| Q5          | Рассуждение в целом начато правильно, но итогового ответа нет | Описан неверный алгоритм: множество уникальных элементов не решает задачу поиска дубликатов | Корректный |  2/6 |
| Q8          | Ошибка: Маша и Ира одновременно названы программистами        | Не выполнена фильтрация по `count > 1`, фактически описан вывод всех ключей                 | Корректный |  2/6 |


### Анализ результатов

По производительности явной зависимости «меньше бит → выше скорость» в эксперименте не получилось. Q5 показал максимальную скорость обычной генерации — **3.17 tok/s**, Q8 — **2.96 tok/s**, а Q4 оказался заметно медленнее — **1.55 tok/s**. На длинном запросе, наоборот, быстрее всего отработал Q8 — **220.65 с**, затем Q4 — **242.62 с**, а Q5 — **297.36 с**.

Время загрузки также не изменялось монотонно: Q5 загрузился быстрее всего (**8.35 с**), затем Q8 (**12.87 с**) и Q4 (**15.86 с**).

Замеры RAM оказались нестабильными: Q4 и Q8 показали примерно одинаковое увеличение памяти, а Q5 — меньшее. Поэтому по одному последовательному запуску процесса эти значения нельзя считать точной оценкой реального потребления памяти каждой квантизацией.

По качеству существенного преимущества более высокой точности не обнаружено. Все три версии хорошо выполнили пересказ, но допустили ошибки в логической задаче и задаче на генерацию кода. На этой небольшой модели разница между Q4, Q5 и Q8 оказалась меньше общего ограничения качества самой модели.

### Вывод

В данном эксперименте увеличение точности квантизации от Q4 до Q8 **не привело к заметному улучшению качества ответов**, а скорость также не изменялась монотонно. Поэтому практический выбор квантизации следует делать с учётом доступной памяти и реальных замеров на целевой нагрузке. Q4 обычно интересен прежде всего экономией памяти, Q5 представляет промежуточный вариант, а Q8 сохраняет большую точность весов ценой большего размера модели.
